# Inspect Saved Original Step 0

这个 notebook 仿照 `deploy_debug/inspect_saved_single_maskaware_action_step0.ipynb`，但不 import `eval_original.py`，只内嵌最小必要函数，在训练机 0 号卡上重跑 original 的 step0 预测。


In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import cv2
import numpy as np
import open3d as o3d
import plotly.graph_objects as go
import plotly.io as pio
import torch
import yaml
import torchvision
from easydict import EasyDict as edict

WORKSPACE_ROOT = Path('/home/haoxiang/rise2_mask_aware')
for p in [WORKSPACE_ROOT, WORKSPACE_ROOT / 'airexo', WORKSPACE_ROOT / 'easyrobot']:
    p_str = str(p)
    if p_str not in sys.path:
        sys.path.insert(0, p_str)

os.chdir(WORKSPACE_ROOT)
print('cwd =', os.getcwd())
from policy import RISE2
from dataset.projector import SingleArmProjector
from dataset.data_utils import ImageProcessor, resize_image
from utils.training import set_seed

pio.renderers.default = 'notebook_connected'
torch.cuda.set_device(0)
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('device =', device)


In [ ]:
CAPTURE_ROOT = Path('/data/haoxiang/data/deploy_debug_260419/deploy_capture_original/capture_20260419_095757')
STEP_ID = 20
STEP_STEM = f'step_{STEP_ID:06d}'
CONFIG_PATH = WORKSPACE_ROOT / 'configs/single_foar_purplebox_original.yaml'
CKPT_PATH = Path('/data/haoxiang/logs/single_foar_purplebox_original/policy_last.ckpt')
CALIB_PATH = Path('/data/haoxiang/data/zihao_foar2/flip_0326_purple_box/calib/rise2_calib_single_foar_purplebox.npy')

META_PATH = CAPTURE_ROOT / 'meta.json'
RGB_PATH = CAPTURE_ROOT / 'rgb' / f'{STEP_STEM}.png'
DEPTH_PATH = CAPTURE_ROOT / 'depth' / f'{STEP_STEM}.png'
ACTION_PATH = CAPTURE_ROOT / 'actions' / f'{STEP_STEM}.npy'
PROPRIO_PATH = CAPTURE_ROOT / 'proprio' / f'{STEP_STEM}.npy'
JOINT_PATH = CAPTURE_ROOT / 'joint' / f'{STEP_STEM}.npy'
POINT_MAX = 120000
VIS_TCP_RADIUS = 0.006

for p in [META_PATH, RGB_PATH, DEPTH_PATH, ACTION_PATH, PROPRIO_PATH, JOINT_PATH, CONFIG_PATH, CKPT_PATH, CALIB_PATH]:
    assert p.exists(), p

meta = json.loads(META_PATH.read_text(encoding='utf-8'))
meta


In [ ]:
rgb = cv2.cvtColor(cv2.imread(str(RGB_PATH), cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
depth = cv2.imread(str(DEPTH_PATH), cv2.IMREAD_UNCHANGED)
saved_action = np.load(ACTION_PATH)
proprio = np.load(PROPRIO_PATH)
joint = np.load(JOINT_PATH)
print('saved_action shape', saved_action.shape, saved_action)
print('proprio shape', proprio.shape)
print('joint shape', joint.shape)


In [ ]:
def create_point_cloud(colors, depths, cam_intrinsics, config, depth_scale=1000.0, rescale_factor=1.0):
    h, w = depths.shape
    fx, fy = cam_intrinsics[0, 0] * rescale_factor, cam_intrinsics[1, 1] * rescale_factor
    cx, cy = cam_intrinsics[0, 2] * rescale_factor, cam_intrinsics[1, 2] * rescale_factor
    color_o3d = o3d.geometry.Image(colors.astype(np.uint8))
    depth_o3d = o3d.geometry.Image(depths.astype(np.float32))
    camera_intrinsics = o3d.camera.PinholeCameraIntrinsic(width=w, height=h, fx=fx, fy=fy, cx=cx, cy=cy)
    rgbd = o3d.geometry.RGBDImage.create_from_color_and_depth(color_o3d, depth_o3d, depth_scale, convert_rgb_to_intensity=False)
    cloud = o3d.geometry.PointCloud.create_from_rgbd_image(rgbd, camera_intrinsics)
    bbox3d = o3d.geometry.AxisAlignedBoundingBox(config.deploy.workspace.min, config.deploy.workspace.max)
    cloud = cloud.crop(bbox3d)
    cloud = cloud.voxel_down_sample(config.data.voxel_size)
    return cloud

def create_input(colors, depths, cam_intrinsics, config, depth_scale=1000.0, rescale_factor=1.0):
    cloud = create_point_cloud(colors, depths, cam_intrinsics, config, depth_scale=depth_scale, rescale_factor=rescale_factor)
    points = np.asarray(cloud.points)
    coords = np.ascontiguousarray(points / config.data.voxel_size, dtype=np.int32)
    return coords, points, cloud

def create_batch(coords, points):
    import MinkowskiEngine as ME
    coords_batch, feats_batch = ME.utils.sparse_collate([coords], [points.astype(np.float32)])
    return coords_batch, feats_batch

def process_state(state, config, to_control=True):
    state = np.asarray(state).copy()
    if to_control:
        state[..., 0:3] = (state[..., 0:3] + 1) / 2.0 * (config.data.normalization.trans_max - config.data.normalization.trans_min) + config.data.normalization.trans_min
        state[..., 9] = (state[..., 9] + 1) / 2.0 * config.data.normalization.max_gripper_width
    else:
        state[..., 0:3] = (state[..., 0:3] - config.data.normalization.trans_min) / (config.data.normalization.trans_max - config.data.normalization.trans_min) * 2.0 - 1
        state[..., 9] = state[..., 9] / config.data.normalization.max_gripper_width * 2.0 - 1
    return state


In [ ]:
with open(CONFIG_PATH, 'r', encoding='utf-8') as f:
    config = edict(yaml.load(f, Loader=yaml.FullLoader))
config.data.normalization.trans_min = np.asarray(config.data.normalization.trans_min)
config.data.normalization.trans_max = np.asarray(config.data.normalization.trans_max)
set_seed(config.deploy.seed)
projector = SingleArmProjector(str(CALIB_PATH), meta['camera_serial'])
policy = RISE2(num_action=config.data.num_action, obs_feature_dim=config.model.obs_feature_dim, cloud_enc_dim=config.model.cloud_enc_dim, image_enc_dim=config.model.image_enc_dim, action_dim=10, hidden_dim=config.model.hidden_dim, nheads=config.model.nheads, num_attn_layers=config.model.num_attn_layers, dim_feedforward=config.model.dim_feedforward, dropout=config.model.dropout, image_enc=config.model.image_enc, interp_fn_mode=config.model.interp_fn_mode, image_enc_finetune=config.model.image_enc_finetune, image_enc_dtype=config.model.image_enc_dtype).to(device)
policy.load_state_dict(torch.load(CKPT_PATH, map_location=device), strict=False)
policy.eval()
image_enc = config.model.image_enc
if image_enc == 'resnet18':
    img_size = config.data.aligner.img_size_resnet
    img_coord_size = config.data.aligner.img_coord_size_resnet
elif image_enc.startswith('dinov2'):
    img_size = config.data.aligner.img_size_dinov2
    img_coord_size = config.data.aligner.img_coord_size_dinov2
elif image_enc.startswith('dinov3'):
    img_size = config.data.aligner.img_size_dinov3
    img_coord_size = config.data.aligner.img_coord_size_dinov3
else:
    raise ValueError(image_enc)
image_processor = ImageProcessor(img_size=img_size, img_coord_size=img_coord_size, voxel_size=config.data.voxel_size, img_mean=config.data.normalization.img_mean, img_std=config.data.normalization.img_std)
intr_data = np.load(CALIB_PATH, allow_pickle=True)
if isinstance(intr_data, np.ndarray) and intr_data.shape == ():
    intr_data = intr_data.item()
intrinsic = np.asarray(intr_data['intrinsics'][meta['camera_serial']], dtype=np.float32)
coords, points, cloud = create_input(rgb, depth, cam_intrinsics=intrinsic, config=config, depth_scale=1000.0, rescale_factor=1.0)
image_coords = image_processor.get_image_coordinates(depth, intrinsic, 1000.0)
colors_t, image_coords_t = image_processor.preprocess_images(rgb, image_coords)
print('cloud points', points.shape)


In [ ]:
import MinkowskiEngine as ME
coords_batch, feats_batch = create_batch(coords, points)
cloud_data = ME.SparseTensor(feats_batch.to(device), coords_batch.to(device))
colors_model = colors_t.unsqueeze(0).to(device)
image_coords_model = image_coords_t.unsqueeze(0).to(device)
with torch.inference_mode():
    pred_raw_action = policy(cloud_data, colors_model, image_coords_model, actions=None).squeeze(0).cpu().numpy()
pred_action_camera = process_state(pred_raw_action.copy(), config, to_control=True)
saved_action_camera = projector.project_tcp_to_camera_coord(saved_action[:9], rotation_rep='rotation_6d')
current_tcp_camera = projector.project_tcp_to_camera_coord(proprio[:9], rotation_rep='rotation_6d')
print('pred_action_camera shape =', pred_action_camera.shape)


In [ ]:
def add_tcp_marker(fig, tcp_pose, color, radius=0.006, size=2.2, opacity=0.95):
    center = np.asarray(tcp_pose[:3], dtype=np.float64)
    offsets = np.array([[0.0,0.0,0.0],[radius,0.0,0.0],[-radius,0.0,0.0],[0.0,radius,0.0],[0.0,-radius,0.0],[0.0,0.0,radius],[0.0,0.0,-radius]])
    pts = center[None, :] + offsets
    fig.add_trace(go.Scatter3d(x=pts[:,0], y=pts[:,1], z=pts[:,2], mode='markers', marker=dict(size=size, color=color, opacity=opacity), showlegend=False))
pc_points = np.asarray(cloud.points)
pc_colors = np.asarray(cloud.colors)
if pc_points.shape[0] > POINT_MAX:
    idx = np.linspace(0, pc_points.shape[0] - 1, POINT_MAX).astype(np.int64)
    pc_points = pc_points[idx]
    pc_colors = pc_colors[idx]
fig = go.Figure()
fig.add_trace(go.Scatter3d(x=pc_points[:,0], y=pc_points[:,1], z=pc_points[:,2], mode='markers', marker=dict(size=1.5, color=pc_colors, opacity=0.55), name='original_vis_point_cloud'))
add_tcp_marker(fig, current_tcp_camera, 'deepskyblue', radius=0.007, size=2.8, opacity=1.0)
for raw_tcp in pred_action_camera:
    add_tcp_marker(fig, raw_tcp, 'dimgray', radius=0.006, size=2.0, opacity=0.9)
add_tcp_marker(fig, saved_action_camera, 'dimgray', radius=0.006, size=2.4, opacity=1.0)
fig.update_layout(title=f'Original deploy-style vis replay, step {STEP_ID}', scene=dict(xaxis_title='X', yaxis_title='Y', zaxis_title='Z', aspectmode='data'), width=1250, height=920, showlegend=False)
fig
